# Fine-tune NeuTTS Nano to a new language

[NeuTTS Nano](https://huggingface.co/neuphonic/neutts-nano) is a small TTS LM that emits NeuCodec audio tokens conditioned on phonemes. Out of the box it speaks English. This notebook adapts it to a **new language** by fine-tuning on a dataset of `(text, NeuCodec codes)` pairs in that language.

The recipe is language-agnostic — change `PHONEMIZER_LANG`, swap the dataset, and the same flow works for French, German, Spanish, ... Italian is used as the worked example because a pre-encoded dataset is already on the Hub.

What you'll learn:
- Building the NeuTTS phonemizer with eSpeak for a new language
- Adding an optional `<|LANG|>` token so a multilingual fine-tune can disambiguate languages
- Constructing the chat-template prompt the model expects (`<|TEXT_PROMPT_START|>` ... `<|SPEECH_GENERATION_START|>` ...)
- Masking labels so loss is only computed on speech tokens, never on prompt or padding
- Training with TRL's `SFTTrainer` and evaluating with Whisper

For the CLI version of this flow, see the sibling scripts in this folder: `finetune_neutts_nano.py` (training), `encode_dataset.py` (data prep), `generate_samples.py` (inference).

## Install dependencies

`espeak-ng` is the backend the `phonemizer` package shells out to.

In [ ]:
!apt-get -qq update && apt-get -qq install -y espeak-ng
!pip install -q "transformers>=4.45" "trl>=0.12" "accelerate>=0.34" "datasets>=2.20" omegaconf phonemizer huggingface_hub soundfile librosa neucodec

## (Optional) Hub login

Only required if you plan to push your fine-tuned model. Skip if you're just training locally.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## Configuration

All the knobs in one place. Flip `SMOKE_TEST` off when you're ready for a real run.

In [ ]:
SMOKE_TEST = True

MODEL_CHECKPOINT = "neuphonic/neutts-nano"
PHONEMIZER_LANG  = "it"         # any eSpeak language code: fr-fr, de, es, pt, ...
LANGUAGE_TOKEN   = "<|IT|>"     # "" to skip; useful for multilingual fine-tunes
ENCODED_DATASET  = "Steveeeeeeen/yodas-granary-it-neucodec-150k"
SPLIT            = "train"

MAX_SEQ_LEN     = 2048
LR              = 2e-5
MAX_STEPS       = 3000
PER_DEVICE_BS   = 64
WARMUP_RATIO    = 0.03

OUTPUT_DIR    = "./neutts-nano-finetuned"
PUSH_TO_HUB   = False
HUB_MODEL_ID  = None  # e.g. "your-user/neutts-nano-italian"

if SMOKE_TEST:
    SPLIT = "train[:64]"
    MAX_STEPS = 10
    PER_DEVICE_BS = 2

print(f"smoke_test={SMOKE_TEST} | language={PHONEMIZER_LANG} | steps={MAX_STEPS} | bs={PER_DEVICE_BS}")

## Load the pre-encoded dataset

The trainer expects two columns:

- `text` — the transcript, in the target language
- `codes` — a flat list of NeuCodec token IDs (each in `[0, 65536)`) for the corresponding audio

`Steveeeeeeen/yodas-granary-it-neucodec-150k` is a 150k-sample Italian dataset already in this format. If you want to encode your own audio, see the optional section below.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(ENCODED_DATASET, split=SPLIT)
print(raw_dataset)
print(raw_dataset.column_names)

sample = raw_dataset[0]
print()
print("text:", sample["text"])
print("codes (first 20):", sample["codes"][:20])
print("codes length:", len(sample["codes"]), "  (NeuCodec ~= 50 tokens/sec at 24 kHz)")

### (Optional) Encode your own audio

If you have raw audio with transcripts and want to build a new `(text, codes)` dataset, use the sibling `encode_dataset.py` script. Leaving this commented because the cell above already has data to train on.

```bash
python encode_dataset.py \
    --source-dataset nvidia/Granary \
    --source-config  it_voxpopuli \
    --source-split   asr \
    --max-samples    1000 \
    --output-dataset your-user/granary-it-neucodec
```

## Load tokenizer + model

NeuTTS Nano is a causal LM whose vocabulary contains the usual text tokens plus a fixed set of `<|speech_N|>` codec tokens (one per NeuCodec entry) and special separators:

- `<|TEXT_PROMPT_START|>` ... `<|TEXT_PROMPT_END|>` — wrap the phoneme prompt
- `<|SPEECH_GENERATION_START|>` ... `<|SPEECH_GENERATION_END|>` — wrap the audio side

If we want a `<|IT|>` (or `<|FR|>`, ...) language tag, we add it to the tokenizer and resize the model embeddings.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForCausalLM.from_pretrained(MODEL_CHECKPOINT, torch_dtype="auto")

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
tokenizer.padding_side = "right"

for tok in ["<|TEXT_PROMPT_START|>", "<|TEXT_PROMPT_END|>",
            "<|SPEECH_GENERATION_START|>", "<|SPEECH_GENERATION_END|>",
            "<|speech_0|>", "<|speech_42|>"]:
    print(f"{tok!r:35s} -> id {tokenizer.convert_tokens_to_ids(tok)}")

if LANGUAGE_TOKEN and tokenizer.convert_tokens_to_ids(LANGUAGE_TOKEN) == tokenizer.unk_token_id:
    tokenizer.add_tokens([LANGUAGE_TOKEN])
    if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
        model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
    print(f"Added {LANGUAGE_TOKEN!r}; new vocab size = {len(tokenizer)}")


## Set up the phonemizer

`phonemizer` uses eSpeak-NG to turn text into IPA phonemes. The exact backend settings here match what the training script uses, so train and inference distributions agree.

A quick sanity print confirms that eSpeak Italian is actually installed.

In [ ]:
import phonemizer

g2p = phonemizer.backend.EspeakBackend(
    language=PHONEMIZER_LANG,
    preserve_punctuation=True,
    with_stress=True,
    words_mismatch="ignore",
    language_switch="remove-flags",
)

example_text = sample["text"]
example_phones = " ".join(g2p.phonemize([example_text])[0].split())
print("text:    ", example_text)
print("phonemes:", example_phones)

## Build one training example end-to-end

This is the teaching part of the notebook. Before we map a preprocessing function over the whole dataset, let's walk through what one row looks like, step by step.

The model is trained to take **phonemes in, codec tokens out**, formatted as a single chat turn:

```
user: Convert the text to speech:<|TEXT_PROMPT_START|>{phonemes}<|TEXT_PROMPT_END|>
assistant:<|SPEECH_GENERATION_START|>{codec_tokens}<|SPEECH_GENERATION_END|>
```

We mask labels so cross-entropy only fires on the assistant side, from `<|SPEECH_GENERATION_START|>` onward.

In [ ]:
IGNORE_INDEX = -100

text = sample["text"]
codes = sample["codes"]

# 1. Phonemize.
phones = " ".join(g2p.phonemize([text])[0].split())
# 2. Optionally prepend a language tag.
if LANGUAGE_TOKEN:
    phones = f"{LANGUAGE_TOKEN} {phones}"
print("phones:", phones[:120], "...")

# 3. Format codec ids as <|speech_N|> tokens.
codes_str = "".join(f"<|speech_{i}|>" for i in codes)
print("codes_str (first 80 chars):", codes_str[:80], "...")

# 4. Assemble the chat template the trainer expects.
chat = (
    f"user: Convert the text to speech:<|TEXT_PROMPT_START|>{phones}<|TEXT_PROMPT_END|>\n"
    f"assistant:<|SPEECH_GENERATION_START|>{codes_str}<|SPEECH_GENERATION_END|>"
)

# 5. Tokenize and pad.
ids = tokenizer.encode(chat)
if len(ids) < MAX_SEQ_LEN:
    ids = ids + [tokenizer.pad_token_id] * (MAX_SEQ_LEN - len(ids))
else:
    ids = ids[:MAX_SEQ_LEN]
input_ids = torch.tensor(ids, dtype=torch.long)

# 6. Build labels: mask everything before <|SPEECH_GENERATION_START|>, supervise after.
speech_gen_start = tokenizer.convert_tokens_to_ids("<|SPEECH_GENERATION_START|>")
labels = torch.full_like(input_ids, IGNORE_INDEX)
start_idx = (input_ids == speech_gen_start).nonzero(as_tuple=True)[0]
if len(start_idx):
    labels[start_idx[0]:] = input_ids[start_idx[0]:]

attention_mask = (input_ids != tokenizer.pad_token_id).long()

print()
print(f"input_ids length     : {len(input_ids)}")
print(f"supervised positions : {(labels != IGNORE_INDEX).sum().item()}  (only the assistant codec span)")
print(f"first supervised tok : {tokenizer.convert_ids_to_tokens(int(input_ids[start_idx[0]]))!r}")

## Map preprocessing over the full dataset

These are the exact `data_filter` and `preprocess_sample` from `finetune_neutts_nano.py`, dropped inline so the notebook stands alone and the recipe is transparent.

In [ ]:
import re
from functools import partial

ACRONYM = re.compile(r"(?:[a-zA-Z]\.){2,}")
ACRONYM_NO_PERIOD = re.compile(r"(?:[A-Z]){2,}")

def data_filter(s):
    t = s["text"]
    if not t: return False
    if re.search(r"\d", t): return False
    if re.search(ACRONYM, t) or re.search(ACRONYM_NO_PERIOD, t): return False
    if t[-1] not in ".,?!": return False
    if "£" in t or "$" in t: return False
    return True


def preprocess_sample(s, tokenizer, max_len, g2p, language_token=""):
    speech_gen_start = tokenizer.convert_tokens_to_ids("<|SPEECH_GENERATION_START|>")
    phones = g2p.phonemize([s["text"]])
    if not phones or not phones[0]:
        return None
    phones = " ".join(phones[0].split())
    if language_token:
        phones = f"{language_token} {phones}"
    codes_str = "".join(f"<|speech_{i}|>" for i in s["codes"])
    chat = (
        f"user: Convert the text to speech:<|TEXT_PROMPT_START|>{phones}<|TEXT_PROMPT_END|>\n"
        f"assistant:<|SPEECH_GENERATION_START|>{codes_str}<|SPEECH_GENERATION_END|>"
    )
    ids = tokenizer.encode(chat)
    if len(ids) < max_len:
        ids = ids + [tokenizer.pad_token_id] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    input_ids = torch.tensor(ids, dtype=torch.long)
    labels = torch.full_like(input_ids, IGNORE_INDEX)
    start_idx = (input_ids == speech_gen_start).nonzero(as_tuple=True)[0]
    if len(start_idx):
        labels[start_idx[0]:] = input_ids[start_idx[0]:]
    attn = (input_ids != tokenizer.pad_token_id).long()
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attn}


_preprocess = partial(
    preprocess_sample,
    tokenizer=tokenizer,
    max_len=MAX_SEQ_LEN,
    g2p=g2p,
    language_token=LANGUAGE_TOKEN,
)

print(f"Rows before filter: {len(raw_dataset)}")
filtered = raw_dataset.filter(data_filter)
print(f"Rows after  filter: {len(filtered)}")
train_dataset = filtered.map(_preprocess, remove_columns=["text", "codes"], num_proc=1 if SMOKE_TEST else 4)
train_dataset = train_dataset.filter(lambda s: s["input_ids"] is not None)
print(f"Rows after  preprocessing: {len(train_dataset)}")

## Train with `SFTTrainer`

These training arguments mirror `finetune_neutts_nano.py` and `config_yodas_it.yaml`. Two knobs to call out:

- **`gradient_checkpointing`**: keep `True` on Colab to fit the model. On an 80 GB GPU you can set it to `False` and bump `per_device_train_batch_size` for higher throughput.
- **`torch_compile`**: on for production runs; off for smoke tests so the first step doesn't pay the compile cost.

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
from transformers import default_data_collator

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    do_train=True,
    learning_rate=LR,
    max_steps=MAX_STEPS,
    bf16=True,
    per_device_train_batch_size=PER_DEVICE_BS,
    warmup_ratio=WARMUP_RATIO,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    save_steps=max(MAX_STEPS, 100),
    logging_steps=1 if SMOKE_TEST else 25,
    save_strategy="no" if SMOKE_TEST else "steps",
    dataloader_drop_last=True,
    remove_unused_columns=False,
    torch_compile=not SMOKE_TEST,
    dataloader_num_workers=2 if SMOKE_TEST else 8,
    report_to="none",
    loss_type="chunked_nll",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer_kwargs = {}
sig = inspect.signature(SFTTrainer.__init__)
if "processing_class" in sig.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sig.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=default_data_collator,
    **trainer_kwargs,
)

trainer.train()

## Save (and optionally push) the fine-tuned model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

if PUSH_TO_HUB and HUB_MODEL_ID:
    from huggingface_hub import create_repo, upload_folder
    create_repo(HUB_MODEL_ID, repo_type="model", exist_ok=True)
    upload_folder(repo_id=HUB_MODEL_ID, repo_type="model", folder_path=OUTPUT_DIR, path_in_repo=".")
    print(f"Pushed to https://huggingface.co/{HUB_MODEL_ID}")

## Voice-clone inference

NeuTTS Nano can clone a voice zero-shot if we seed it with a reference clip's phonemes + codec tokens, then ask it to continue with the *target* phonemes. The decoder will produce target speech in the reference voice.

Prompt format (same one used at training time):

```
user: Convert the text to speech:<|TEXT_PROMPT_START|>{ref_phonemes} {target_phonemes}<|TEXT_PROMPT_END|>
assistant:<|SPEECH_GENERATION_START|>{ref_codec_tokens}    <-- model continues from here
```

In [ ]:
import re as _re
from neucodec import NeuCodec

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()
codec = NeuCodec.from_pretrained("neuphonic/neucodec").eval().to(device)

# Pick a reference voice from the dataset.
ref_row   = raw_dataset[0]
ref_text  = ref_row["text"]
ref_codes = [int(c) for c in ref_row["codes"]]

target_text = "Buongiorno, oggi è una bellissima giornata e voglio raccontarti una storia."

ref_phones    = " ".join(g2p.phonemize([ref_text])[0].split())
target_phones = " ".join(g2p.phonemize([target_text])[0].split())
prompt_text   = f"{ref_phones} {target_phones}"
if LANGUAGE_TOKEN:
    prompt_text = f"{LANGUAGE_TOKEN} {prompt_text}"
speech_token_text = "".join(f"<|speech_{i}|>" for i in ref_codes)

prompt = (
    f"user: Convert the text to speech:<|TEXT_PROMPT_START|>{prompt_text}<|TEXT_PROMPT_END|>\n"
    f"assistant:<|SPEECH_GENERATION_START|>{speech_token_text}"
)
input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)

speech_end_id = tokenizer.convert_tokens_to_ids("<|SPEECH_GENERATION_END|>")
with torch.no_grad():
    out = model.generate(
        input_ids,
        max_new_tokens=900,
        min_new_tokens=50,
        do_sample=True, temperature=0.9, top_k=50,
        eos_token_id=speech_end_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

# Extract the <|speech_N|> tokens the model just generated and decode to a waveform.
SPEECH_TOKEN_RE = _re.compile(r"<\|speech_(\d+)\|>")
new_token_ids = out[0, input_ids.shape[-1]:].detach().cpu().tolist()
speech_ids = []
for tid in new_token_ids:
    if tid == speech_end_id:
        break
    piece = tokenizer.convert_ids_to_tokens(int(tid))
    m = SPEECH_TOKEN_RE.match(piece)
    if m:
        speech_ids.append(int(m.group(1)))

print(f"Generated {len(speech_ids)} codec tokens.")
codes_tensor = torch.tensor(speech_ids, dtype=torch.long, device=device).view(1, 1, -1)
with torch.no_grad():
    wav = codec.decode_code(codes_tensor).detach().cpu().squeeze().float().numpy()

import soundfile as sf
from IPython.display import Audio, display
out_path = f"neutts_{PHONEMIZER_LANG}_generated.wav"
sf.write(out_path, wav, 24_000)
print(f"Wrote {out_path}  ({len(wav)/24000:.2f}s)")
display(Audio(out_path))

## Whisper large-v3 sanity check

Transcribe the generated audio with Whisper in the target language and compare to the prompt. Word overlap is a quick eyeball metric — for a real eval you'd compute WER on a held-out set.

> **Note**: with `SMOKE_TEST=True` the model only ran for 10 steps, so the generated audio will be noise and Whisper will return `...` or an empty string. That's expected — flip `SMOKE_TEST=False` and run the full 3000 steps to get intelligible Italian.

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import librosa

WHISPER_ID = "openai/whisper-large-v3"
processor = AutoProcessor.from_pretrained(WHISPER_ID)
whisper = AutoModelForSpeechSeq2Seq.from_pretrained(WHISPER_ID, torch_dtype=torch.float16 if device.type == "cuda" else torch.float32).to(device).eval()

wav_16k = librosa.resample(wav, orig_sr=24_000, target_sr=16_000)
inputs = processor(wav_16k, sampling_rate=16_000, return_tensors="pt").to(device, whisper.dtype if device.type == "cuda" else torch.float32)
with torch.no_grad():
    pred_ids = whisper.generate(
        **inputs,
        language=PHONEMIZER_LANG,
        task="transcribe",
        max_new_tokens=200,
    )
transcript = processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()

print("target    :", target_text)
print("whisper   :", transcript)

def words(s):
    return [w.lower().strip(".,?!") for w in s.split() if w]
overlap = len(set(words(target_text)) & set(words(transcript))) / max(1, len(set(words(target_text))))
print(f"word overlap: {overlap:.0%}")

## Adapting to another language

To run the same recipe on a different language:

1. Change `PHONEMIZER_LANG` to the relevant [eSpeak code](https://github.com/espeak-ng/espeak-ng/blob/master/docs/languages.md) (e.g. `"fr-fr"`, `"de"`, `"es"`).
2. Set `LANGUAGE_TOKEN` to `"<|FR|>"` / `"<|DE|>"` / ... if you want an explicit language tag, otherwise leave it `""`.
3. Point `ENCODED_DATASET` at a Hub dataset with `text` + `codes` in the target language. If one doesn't exist, run `encode_dataset.py` on a public ASR dataset (Granary, CML-TTS, VoxPopuli, ...) to make one.

For larger production runs, edit `config_yodas_it.yaml` accordingly and use `finetune_neutts_nano.py` from the CLI — it's the same flow as this notebook, just config-driven.